# DPO: Direct Preference Optimization

DPO (Rafailov et al. 2023) showed that the RLHF objective can be optimized directly without training a reward model or running an RL loop. It replaced PPO as the dominant alignment method within months of publication.

## The Key Insight: RLHF Has a Closed-Form Solution

The constrained RLHF objective — maximize reward while staying close to the reference policy — has an analytical optimal policy:

```
π*(y|x) ∝ π_ref(y|x) * exp(r(x,y) / β)
```

DPO's insight: we can rearrange this to express the reward in terms of the policy and reference policy. Substituting into the Bradley-Terry preference model and simplifying, the preference loss becomes:

```
L_DPO = -log σ(β * log[π_θ(y_w|x)/π_ref(y_w|x)] - β * log[π_θ(y_l|x)/π_ref(y_l|x)])
```

This is a binary cross-entropy loss over preference pairs that trains the policy directly, with no separate reward model and no RL loop. The implicit reward is `β * log[π_θ(y|x)/π_ref(y|x)]`.

In [ ]:
import math
import random
from dataclasses import dataclass

@dataclass
class DPOBatch:
    prompt: str
    chosen: str
    rejected: str
    log_prob_chosen_policy: float
    log_prob_rejected_policy: float
    log_prob_chosen_ref: float
    log_prob_rejected_ref: float

def dpo_loss(batch: DPOBatch, beta: float = 0.1) -> float:
    chosen_logratios = batch.log_prob_chosen_policy - batch.log_prob_chosen_ref
    rejected_logratios = batch.log_prob_rejected_policy - batch.log_prob_rejected_ref
    logits = beta * (chosen_logratios - rejected_logratios)
    loss = -math.log(1.0 / (1.0 + math.exp(-logits)) + 1e-8)
    return loss

def implicit_reward(log_prob_policy: float, log_prob_ref: float, beta: float = 0.1) -> float:
    return beta * (log_prob_policy - log_prob_ref)

# Demonstrate DPO loss dynamics
batches = [
    DPOBatch('What is 2+2?', 'It is 4.', 'I am not sure.',
             log_prob_chosen_policy=-0.5, log_prob_rejected_policy=-3.0,
             log_prob_chosen_ref=-1.0, log_prob_rejected_ref=-1.5),
    DPOBatch('Explain gravity', 'Gravity is a force...', 'Gravity is complicated.',
             log_prob_chosen_policy=-2.0, log_prob_rejected_policy=-2.1,
             log_prob_chosen_ref=-2.0, log_prob_rejected_ref=-2.0),
]

for b in batches:
    loss = dpo_loss(b)
    r_chosen = implicit_reward(b.log_prob_chosen_policy, b.log_prob_chosen_ref)
    r_rejected = implicit_reward(b.log_prob_rejected_policy, b.log_prob_rejected_ref)
    print(f'Prompt: {b.prompt}')
    print(f'  DPO loss: {loss:.4f}')
    print(f'  Implicit reward (chosen):   {r_chosen:.4f}')
    print(f'  Implicit reward (rejected): {r_rejected:.4f}')
    print()

## DPO vs PPO: Practical Comparison

| Property | PPO | DPO |
|----------|-----|-----|
| Models in memory | 4 (policy, ref, RM, critic) | 2 (policy, ref) |
| Training loop | RL (complex) | Supervised (simple) |
| Reward model needed | Yes | No |
| Hyperparameters | Many (KL, PPO clip, value coef) | Few (β, lr) |
| Stability | Prone to collapse | Stable |
| Online vs offline | Online (generates new samples) | Offline (trains on fixed pairs) |
| Quality ceiling | Slightly higher (can explore) | Slightly lower (no exploration) |

The quality ceiling difference matters in theory but is small in practice. For most alignment tasks, DPO's simplicity and stability make it the better choice.

In [ ]:
class SimpleDPOTrainer:
    def __init__(self, beta: float = 0.1, lr: float = 1e-5, seed: int = 42):
        self.beta = beta
        self.lr = lr
        self.rng = random.Random(seed)
        # Proxy for policy: a single quality scalar
        self.policy_score = 0.0
        self.losses = []

    def _simulate_log_probs(self, text: str, is_policy: bool) -> float:
        base = -len(text.split()) * 0.1
        if is_policy:
            return base + self.policy_score * 0.1 + self.rng.gauss(0, 0.05)
        return base + self.rng.gauss(0, 0.05)  # reference is fixed

    def train_step(self, chosen: str, rejected: str, prompt: str) -> float:
        lp_c_pi = self._simulate_log_probs(chosen, True)
        lp_r_pi = self._simulate_log_probs(rejected, True)
        lp_c_ref = self._simulate_log_probs(chosen, False)
        lp_r_ref = self._simulate_log_probs(rejected, False)
        batch = DPOBatch(prompt, chosen, rejected, lp_c_pi, lp_r_pi, lp_c_ref, lp_r_ref)
        loss = dpo_loss(batch, self.beta)
        self.losses.append(loss)
        # Policy improves by reducing loss
        self.policy_score += self.lr * (1.0 - loss) * 1000
        return loss

trainer = SimpleDPOTrainer(beta=0.1)
pairs = [
    ('The answer is 42.', 'I think the answer might be 42 or possibly other numbers.'),
    ('Photosynthesis converts CO2 and water into glucose using sunlight.', 'Plants do photosynthesis.'),
    ('Sort with list.sort() for in-place or sorted() for a new list.', 'You can sort in Python.'),
]

print('DPO Training Progress:')
for epoch in range(3):
    epoch_loss = 0
    for chosen, rejected in pairs:
        epoch_loss += trainer.train_step(chosen, rejected, 'question')
    print(f'  Epoch {epoch+1}: avg_loss={epoch_loss/len(pairs):.4f}, policy_score={trainer.policy_score:.4f}')

## DPO in Practice: Data Requirements

DPO requires preference pairs: (prompt, chosen_response, rejected_response). Key considerations:

**Preference data sources**:
- Human annotations (most expensive, highest quality)
- AI feedback: use a stronger model to judge two weaker model responses
- Existing datasets: HH-RLHF (Anthropic), UltraFeedback, Orca-DPO

**Filtering considerations**:
- Remove pairs where the chosen and rejected responses are very similar (low signal)
- Remove pairs where the reward gap is below a threshold
- Ensure prompts cover your target distribution

**β tuning**: lower β (0.01-0.1) allows more deviation from reference; higher β (0.5-1.0) keeps the model closer to the original. Start with 0.1.

## DPO Variants and Extensions

**IPO** (Identity Preference Optimization): fixes a theoretical issue with DPO where the optimal solution can have infinite reward difference. More stable on noisy data.

**ORPO** (Odds Ratio Preference Optimization): eliminates the reference model entirely by incorporating the preference objective directly into the SFT loss. Covered in the next notebook.

**SimPO** (Simple Preference Optimization): uses response length-normalized rewards to reduce the verbosity bias inherent in log-probability-based rewards.

**RPO** (Robust Preference Optimization): adds a supervised loss on the chosen responses alongside DPO to prevent degradation of generation quality.